In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os

import cv2
import kagglehub
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.transforms.functional as TF


In [ ]:
# ============================================================
# 2. Configuration / Paths / Device
# ============================================================
path = kagglehub.competition_download('aptos2019-blindness-detection')
image_dir = os.path.join(path, "train_images")
csv_path = os.path.join(path, "train.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Path to competition files:", path)
print("Using:", device)


In [ ]:
# ============================================================
# 3. APTOS Dataset and Preprocessing
# ============================================================
df = pd.read_csv(csv_path)

print("\nOriginal dataset:")
print(df.head())
print("\nClass distribution:")
print(df["diagnosis"].value_counts().sort_index())

# Sample 30% of the complete dataset, then make a stratified train/validation split.
sample_fraction = 0.30
sample_df, _ = train_test_split(
    df,
    test_size=1 - sample_fraction,
    stratify=df["diagnosis"],
    random_state=42
)
sample_df = sample_df.reset_index(drop=True)

train_df, val_df = train_test_split(
    sample_df,
    test_size=0.2,
    stratify=sample_df["diagnosis"],
    random_state=42
)

print("\nSample size:", len(sample_df))
print("Train:", len(train_df))
print("Validation:", len(val_df))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class APTOSDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row["id_code"]
        label = int(row["diagnosis"])
        image_path = os.path.join(self.image_dir, image_id + ".png")
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = APTOSDataset(train_df, image_dir, train_transform)
val_dataset = APTOSDataset(val_df, image_dir, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [ ]:
# ============================================================
# 4. Image Classification Model (Swin-T)
# ============================================================
model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=True,
    num_classes=5
)
model = model.to(device)


In [ ]:
# ============================================================
# 5. Focal Loss and Classification Training
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    epoch_loss = running_loss / total
    epoch_accuracy = correct / total
    return epoch_loss, epoch_accuracy


train_loss, train_acc = train_one_epoch(
    model, train_loader, criterion, optimizer, device
)
print("Train Loss:", train_loss)
print("Train Accuracy:", train_acc)


In [ ]:
# ============================================================
# 6. IDRiD Segmentation Dataset and Paths
# ============================================================
MA_PATH = "/kaggle/input/datasets/divyanshukj4495/idrid-lesion-segmentation/A. Segmentation/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/1. Microaneurysms"
HE_PATH = "/kaggle/input/datasets/divyanshukj4495/idrid-lesion-segmentation/A. Segmentation/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/2. Haemorrhages"
EX_PATH = "/kaggle/input/datasets/divyanshukj4495/idrid-lesion-segmentation/A. Segmentation/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/3. Hard Exudates"
SE_PATH = "/kaggle/input/datasets/divyanshukj4495/idrid-lesion-segmentation/A. Segmentation/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/4. Soft Exudates"
IDRID_IMG_PATH = "/kaggle/input/datasets/divyanshukj4495/idrid-lesion-segmentation/B. Disease Grading/B. Disease Grading/1. Original Images/a. Training Set"


def load_mask_if_exists(path, filename, shape):
    """Load a lesion mask, returning an empty mask when its annotation is absent."""
    file_path = os.path.join(path, filename)
    if os.path.exists(file_path):
        mask = np.array(Image.open(file_path))
        return mask > 0
    return np.zeros(shape, dtype=bool)


def load_combined_mask(image_id):
    # Use the MA annotation as the reference for image/mask size.
    ma_file = os.path.join(MA_PATH, image_id + "_MA.tif")
    ma = np.array(Image.open(ma_file))
    shape = ma.shape
    he = load_mask_if_exists(HE_PATH, image_id + "_HE.tif", shape)
    ex = load_mask_if_exists(EX_PATH, image_id + "_EX.tif", shape)
    se = load_mask_if_exists(SE_PATH, image_id + "_SE.tif", shape)
    ma = ma > 0
    combined = ma | he | ex | se
    return combined.astype(np.uint8)


class IDRiDSegmentationDataset(Dataset):
    def __init__(self, image_dir):
        self.image_dir = image_dir
        image_files = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith(".jpg")
        ]
        self.image_map = {}
        for f in image_files:
            number = f.split("_")[-1].split(".")[0]
            number = int(number)
            self.image_map[number] = f

        mask_files = [
            f for f in os.listdir(MA_PATH)
            if f.endswith("_MA.tif")
        ]
        self.image_ids = []
        for f in mask_files:
            number = f.split("_")[1]
            number = int(number)
            if number in self.image_map:
                self.image_ids.append(number)
        self.image_ids.sort()
        print("Usable segmentation images:", len(self.image_ids))

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        number = self.image_ids[idx]
        image_name = self.image_map[number]
        image_path = os.path.join(self.image_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        image_id = f"IDRiD_{number:02d}"
        mask = load_combined_mask(image_id)
        mask = Image.fromarray(mask * 255)
        image = image.resize((224, 224))
        mask = mask.resize((224, 224), Image.Resampling.NEAREST)
        image = TF.to_tensor(image)
        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        return image, mask


idrid_dataset = IDRiDSegmentationDataset(IDRID_IMG_PATH)
seg_loader = DataLoader(
    idrid_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


In [ ]:
# ============================================================
# 7. Segmentation Head

# ============================================================
# Auxiliary lesion segmentation head.
model.seg_head = nn.Conv2d(
    model.num_features,
    1,
    kernel_size=1
).to(device)


def image_agent_forward(image):
    image = image.to(device)
    features = model.forward_features(image)
    logits = model.forward_head(features)
    features_seg = features.permute(0, 3, 1, 2)
    lesion_logits = model.seg_head(features_seg)
    lesion_map = F.interpolate(
        lesion_logits,
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )
    return logits, lesion_map


In [ ]:
# ============================================================
# 8. Segmentation Training

# ============================================================
# Retain the notebook's second head initialization before segmentation training.
model.seg_head = nn.Conv2d(
    model.num_features,
    1,
    kernel_size=1
).to(device)

seg_criterion = nn.BCEWithLogitsLoss()
seg_optimizer = torch.optim.AdamW(model.seg_head.parameters(), lr=1e-3)
model.eval()
model.seg_head.train()
num_epochs = 3

for epoch in range(num_epochs):
    total_loss = 0
    for images, masks in seg_loader:
        images = images.to(device)
        masks = masks.to(device)
        with torch.no_grad():
            features = model.forward_features(images)
        features = features.permute(0, 3, 1, 2)
        lesion_logits = model.seg_head(features)
        lesion_logits = F.interpolate(
            lesion_logits,
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        )
        loss = seg_criterion(lesion_logits, masks)
        seg_optimizer.zero_grad()
        loss.backward()
        seg_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(seg_loader)
    print(
        f"Epoch {epoch+1}/{num_epochs} "
        f"| Segmentation Loss: {avg_loss:.4f}"
    )


In [ ]:
# ============================================================
# 9. Grad-CAM
# ============================================================
def generate_gradcam(model, image, target_class=None):
    model.eval()
    image = image.unsqueeze(0).to(device)
    features = model.forward_features(image)
    features.retain_grad()
    logits = model.forward_head(features)
    if target_class is None:
        target_class = logits.argmax(dim=1).item()
    score = logits[0, target_class]
    model.zero_grad()
    score.backward()
    gradients = features.grad
    weights = gradients.mean(dim=(1, 2), keepdim=True)
    cam = (weights * features).sum(dim=-1)
    cam = F.relu(cam)
    cam = cam[0]
    cam = cam - cam.min()
    if cam.max() > 0:
        cam = cam / cam.max()
    cam = F.interpolate(
        cam.unsqueeze(0).unsqueeze(0),
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )
    cam = cam[0, 0].detach().cpu().numpy()
    return cam, target_class, logits.detach().cpu()


In [ ]:
# ============================================================
# 10. Lesion Localization / Lesion Boxes

# ============================================================
# The existing Grad-CAM threshold, morphology, component filtering, and box
# extraction are applied inside image_agent below.


In [ ]:
# ============================================================
# 11. TTA Consistency Check
# ============================================================
def get_tta_images(image):
    """Return the original, horizontal flip, and small rotation of an image."""
    original = image
    flipped = TF.hflip(image)
    rotated = TF.rotate(image, angle=10)
    return [original, flipped, rotated]


def get_grade_prediction(image):
    model.eval()
    with torch.no_grad():
        image = image.unsqueeze(0).to(device)
        features = model.forward_features(image)
        logits = model.forward_head(features)
        probabilities = torch.softmax(logits, dim=1)
        grade = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0, grade].item()
    return grade, confidence


In [ ]:
# ============================================================
# 12. Image Embedding
# ============================================================
def get_image_embedding(image):
    model.eval()
    with torch.no_grad():
        image = image.unsqueeze(0).to(device)
        features = model.forward_features(image)
        # [B, 7, 7, 768] -> [B, 768]
        embedding = features.mean(dim=(1, 2))
    return embedding.squeeze(0).cpu()


In [ ]:
# ============================================================
# 13. Final Image Agent
# ============================================================
def image_agent(image):
    # 1. Classification
    grade, confidence = get_grade_prediction(image)

    # 2. Image embedding
    image_embedding = get_image_embedding(image)

    # 3. Grad-CAM
    cam_result = generate_gradcam(model, image)
    cam = cam_result[0]

    # 4. Lesion boxes
    threshold = np.quantile(cam, 0.5)
    lesion_mask = (cam >= threshold).astype(np.uint8)
    kernel = np.ones((3, 3), np.uint8)
    lesion_mask = cv2.morphologyEx(lesion_mask, cv2.MORPH_OPEN, kernel)
    lesion_mask = cv2.morphologyEx(lesion_mask, cv2.MORPH_CLOSE, kernel)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        lesion_mask, connectivity=8
    )
    boxes = []
    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        if area < 20:
            continue
        boxes.append({
            "x": int(x),
            "y": int(y),
            "width": int(w),
            "height": int(h),
            "area": int(area)
        })

    # 5. Segmentation prediction
    model.eval()
    with torch.no_grad():
        image_batch = image.unsqueeze(0).to(device)
        features = model.forward_features(image_batch)
        features = features.permute(0, 3, 1, 2)
        lesion_logits = model.seg_head(features)
        lesion_logits = F.interpolate(
            lesion_logits,
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        )
        segmentation_mask = torch.sigmoid(lesion_logits)[0, 0].cpu().numpy()

    # 6. TTA
    tta_images = get_tta_images(image)
    tta_grades = []
    tta_confidences = []
    for tta_image in tta_images:
        g, c = get_grade_prediction(tta_image)
        tta_grades.append(g)
        tta_confidences.append(c)
    grade_range = max(tta_grades) - min(tta_grades)
    review_required = grade_range > 1

    return {
        "grade": grade,
        "confidence": confidence,
        "image_embedding": image_embedding,
        "gradcam": cam,
        "lesion_boxes": boxes,
        "segmentation_mask": segmentation_mask,
        "tta_grades": tta_grades,
        "tta_confidences": tta_confidences,
        "grade_range": grade_range,
        "review_required": review_required
    }


In [ ]:
# ============================================================
# 14. Image Agent Test / Demo
# ============================================================
image, _ = idrid_dataset[0]
output = image_agent(image)

print("Grade:", output["grade"])
print("Confidence:", output["confidence"])
print("Embedding:", output["image_embedding"].shape)
print("TTA grades:", output["tta_grades"])
print("Review required:", output["review_required"])
print("Lesion boxes:", len(output["lesion_boxes"]))
print("Segmentation:", output["segmentation_mask"].shape)
print("GradCAM:", output["gradcam"].shape)
